## Setup & Imports

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from pathlib import Path
from matplotlib.ticker import FuncFormatter

# Plot styling
sns.set(style="whitegrid", font_scale=1.2)
sns.set_context("talk")

## Load Data

In [2]:
# ================================
# Load Matches & Teams
# ================================
matches = pd.read_parquet("Datasets/SkillCorner Premier League 24-25 data/matches_clean.parquet")

team_lookup = pd.concat([
    matches[["home_team_id","home_team_name"]].rename(columns={"home_team_id":"team_id","home_team_name":"team_name"}),
    matches[["away_team_id","away_team_name"]].rename(columns={"away_team_id":"team_id","away_team_name":"team_name"})
]).drop_duplicates()

# ================================
# Load Events Data
# ================================
folder = Path("Datasets/SkillCorner Premier League 24-25 data/dynamic_events_pl_24/dynamic")
dfs = [pd.read_parquet(file) for file in folder.glob("*.parquet")]
events = pd.concat(dfs, ignore_index=True)

print(f"Total events: {len(events)}, Unique matches: {events['match_id'].nunique()}")
print(events["event_type"].value_counts().head(10))

# ================================
# Load Player Match Data
# ================================
players_match = pd.read_parquet("Datasets/SkillCorner Premier League 24-25 data/players_match.parquet")

players_match = players_match.rename(columns={
    "id": "player_id",
    "short_name": "player_name",
    "player_role_acronym": "position",
    "player_role_position_group": "position_group"
})

Total events: 1811078, Unique matches: 378
event_type
passing_option        939059
player_possession     362853
on_ball_engagement    326100
off_ball_run          183066
Name: count, dtype: int64


In [3]:
print(f"Total events: {len(events)}, Unique matches: {events['match_id'].nunique()}")
print(events["event_type"].value_counts().head(10))

Total events: 1811078, Unique matches: 378
event_type
passing_option        939059
player_possession     362853
on_ball_engagement    326100
off_ball_run          183066
Name: count, dtype: int64


## Minutes Summary & Main Position

In [4]:
# ================================
# Minutes Summary
# ================================
minutes_summary = (
    players_match.groupby(["player_id","player_name"])
    .agg(
        minutes_tip=("playing_time_total_minutes_tip","sum"),
        minutes_otip=("playing_time_total_minutes_otip","sum"),
        minutes_played=("playing_time_total_minutes_played","sum")
    )
    .reset_index()
)

# ================================
# Determine Main Position
# ================================
position_minutes = (
    players_match.groupby(["player_id","player_name","position","position_group"], as_index=False)
    .agg(minutes_played=("playing_time_total_minutes_played","sum"))
)
position_minutes_sorted = position_minutes.sort_values(["player_id","minutes_played"], ascending=[True,False])

def get_main_position(df):
    main_pos = df.iloc[0]["position"]
    pos_group = df.iloc[0]["position_group"]

    if main_pos == "SUB" and len(df) > 1:
        main_pos = df.iloc[1]["position"]
        pos_group = df.iloc[1]["position_group"]

    if main_pos == "GK":
        pos_group = "Goalkeeper"

    return pd.Series({"main_position": main_pos, "position_group": pos_group})

main_positions = position_minutes_sorted.groupby("player_id", group_keys=False).apply(get_main_position).reset_index()

# Merge final player dataset
players = minutes_summary.merge(main_positions, on="player_id", how="left")
players["minutes_played"] = players["minutes_played"].replace(0, np.nan)

C:\Users\vicky\AppData\Local\Temp\ipykernel_11232\207208563.py:36: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  main_positions = position_minutes_sorted.groupby("player_id", group_keys=False).apply(get_main_position).reset_index()


In [5]:
players.head()

,player_id,player_name,minutes_tip,minutes_otip,minutes_played,main_position,position_group
0,82,A. Cresswell,240.62,280.86,937.72,LCB,Central Defender
1,124,A. Doucouré,649.13,886.57,2792.56,AM,Midfield
2,168,A. Armstrong,395.59,356.80,1381.38,CF,Center Forward
3,192,A. Lallana,125.61,124.60,429.45,LM,Midfield
4,209,A. Smith,436.55,478.56,1753.84,RB,Full Back


# Passing Options & Decision Metrics

In [7]:
passing_options = events[events["event_type"] == "passing_option"].copy()

passing_options = passing_options.rename(columns={
    "player_in_possession_id": "passer_id",
    "player_in_possession_name": "passer_name",
    "player_id": "receiver_id",
    "player_name": "receiver_name"
})

passing_options["pass_value"] = passing_options["xthreat"] * passing_options["xpass_completion"]

# Best option per possession
best_option = (
    passing_options.groupby(["match_id","associated_player_possession_event_id"], as_index=False)
    .agg(best_pass_value=("pass_value","max"))
)
passing_options = passing_options.merge(best_option, on=["match_id","associated_player_possession_event_id"], how="left")

# Keep possessions with >1 option
option_counts = passing_options.groupby(["match_id","associated_player_possession_event_id"]).size().reset_index(name="n_options")
passing_options = passing_options.merge(option_counts, on=["match_id","associated_player_possession_event_id"], how="left")
passing_options = passing_options[passing_options["n_options"] > 1]

# Extract actual passes
chosen_passes = passing_options[passing_options["targeted"] == True].copy()

# Decision quality
chosen_passes["decision_quality"] = chosen_passes["pass_value"] / chosen_passes["best_pass_value"].replace(0,np.nan)
chosen_passes["chose_best"] = chosen_passes["pass_value"] >= chosen_passes["best_pass_value"] * 0.95

# Execution
chosen_passes["actual_completion"] = (chosen_passes["received"] == True).astype(int)
chosen_passes["completion_minus_xpass"] = chosen_passes["actual_completion"] - chosen_passes["xpass_completion"]

# ================================
# Passer Metrics (Per Player)
# ================================
passer_metrics = (
    chosen_passes.groupby(["passer_id","passer_name","team_shortname"])
    .agg(
        passes_attempted=("pass_value","count"),
        avg_decision_quality=("decision_quality","mean"),
        chose_best_rate=("chose_best","mean"),
        chose_best_count=("chose_best","sum"),
        chose_not_best_count=("chose_best", lambda x: (~x).sum()),
        completion_minus_xpass_per_pass=("completion_minus_xpass","mean"),
        completion_minus_xpass_total=("completion_minus_xpass","sum"),
        total_pass_value=("pass_value","sum"),
        avg_pass_value=("pass_value","mean"),
        avg_xthreat=("xthreat","mean"),
        avg_xpass_completion=("xpass_completion","mean")
    )
    .reset_index()
).merge(players, left_on="passer_id", right_on="player_id", how="left")

# ================================
# Receiver Metrics
# ================================
receiver_metrics = (
    passing_options.groupby(["receiver_id","receiver_name","team_shortname"])
    .agg(
        passing_options_available=("receiver_id","count"),
        times_targeted=("targeted","sum"),
        option_selection_rate=("targeted","mean"),
        total_option_value=("pass_value","sum"),
        avg_option_value=("pass_value","mean"),
        total_xthreat_available=("xthreat","sum"),
        avg_xthreat_available=("xthreat","mean"),
        avg_xpass_passing_option_created=("xpass_completion","mean")
    )
    .reset_index()
).merge(players, left_on="receiver_id", right_on="player_id", how="left")

## Off-Ball Run Metrics

In [8]:
off_ball_runs = events[events["event_type"] == "off_ball_run"].copy()
off_ball_runs = off_ball_runs.dropna(axis=1, how='all')

player_offball_metrics = (
    off_ball_runs.groupby(["player_id", "player_name"])
    .agg(
        offball_runs_total=("event_id", "count"),
        offball_distance_total=("distance_covered", "sum"),
        offball_distance_avg=("distance_covered", "mean"),
        offball_speed_avg=("speed_avg", "mean")
    )
    .reset_index()
).merge(players[['player_id', 'minutes_played']], on='player_id', how='left')

## On-ball Engagement

In [6]:
on_ball_engagement = events[events["event_type"] == "on_ball_engagement"].copy()

# Drop columns that are all NaN
on_ball_engagement = on_ball_engagement.dropna(axis=1, how='all')

pd.set_option('display.max_columns', None)  # ensure all columns are visible
on_ball_engagement.head()

,event_id,index,match_id,frame_start,frame_end,frame_physical_start,time_start,time_end,minute_start,second_start,duration,period,attacking_side_id,attacking_side,event_type_id,event_type,event_subtype_id,event_subtype,player_id,player_name,player_position_id,player_position,player_in_possession_id,player_in_possession_name,player_in_possession_position_id,player_in_possession_position,team_id,team_shortname,x_start,y_start,channel_id_start,channel_start,third_id_start,third_start,penalty_area_start,x_end,y_end,channel_id_end,channel_end,third_id_end,third_end,penalty_area_end,associated_player_possession_event_id,associated_player_possession_frame_start,associated_player_possession_frame_end,associated_player_possession_end_type_id,associated_player_possession_end_type,game_state_id,game_state,team_score,opponent_team_score,phase_index,team_possession_loss_in_phase,team_in_possession_phase_type_id,team_in_possession_phase_type,team_out_of_possession_phase_type_id,team_out_of_possession_phase_type,game_interruption_before_id,game_interruption_before,game_interruption_after_id,game_interruption_after,lead_to_shot,lead_to_goal,distance_covered,trajectory_angle,trajectory_direction_id,trajectory_direction,speed_avg,speed_avg_band_id,speed_avg_band,last_defensive_line_x_start,last_defensive_line_x_end,delta_to_last_defensive_line_start,delta_to_last_defensive_line_end,delta_to_last_defensive_line_gain,last_defensive_line_height_start,last_defensive_line_height_end,last_defensive_line_height_gain,end_type_id,end_type,consecutive_on_ball_engagements,player_targeted_id,player_targeted_name,player_targeted_position_id,player_targeted_position,player_targeted_distance_to_goal_start,player_targeted_distance_to_goal_end,player_targeted_angle_to_goal_start,player_targeted_angle_to_goal_end,player_targeted_average_speed,player_targeted_speed_avg_band_id,player_targeted_speed_avg_band,speed_difference,n_teammates_ahead_end,n_teammates_ahead_start,n_player_targeted_opponents_ahead_start,n_player_targeted_opponents_ahead_end,n_player_targeted_teammates_ahead_start,n_player_targeted_teammates_ahead_end,n_player_targeted_teammates_within_5m_start,n_player_targeted_teammates_within_5m_end,n_player_targeted_opponents_within_5m_start,n_player_targeted_opponents_within_5m_end,interplayer_distance_start,interplayer_distance_end,interplayer_distance_min,interplayer_distance_start_physical,close_at_player_possession_start,angle_of_engagement,goal_side_start,goal_side_end,pressing_chain,pressing_chain_length,pressing_chain_end_type_id,pressing_chain_end_type,pressing_chain_index,index_in_pressing_chain,simultaneous_defensive_engagement_same_target,simultaneous_defensive_engagement_same_target_rank,affected_line_breaking_passing_option_id,affected_line_break_id,affected_line_break,affected_line_breaking_passing_option_attempted,affected_line_breaking_passing_option_xthreat,affected_line_breaking_passing_option_dangerous,affected_line_breaking_passing_option_run_subtype_id,affected_line_breaking_passing_option_run_subtype,possession_danger,beaten_by_possession,beaten_by_movement,stop_possession_danger,reduce_possession_danger,force_backward,xloss_player_possession_start,xloss_player_possession_end,xloss_player_possession_max,xshot_player_possession_start,xshot_player_possession_end,xshot_player_possession_max,is_player_possession_start_matched,is_player_possession_end_matched,is_previous_pass_matched,is_pass_reception_matched
4,9_0,4,1650385,62,71,59.0,00:04.9,00:06.1,0,4,1.2,1,1,left_to_right,9,on_ball_engagement,11.0,pressing,13078,M. Mount,15,AM,13068.0,S. Lukić,12.0,LM,31,Manchester U,2.56,-6.42,3,center,2,middle_third,False,6.32,-1.46,3,center,2,middle_third,False,8_1,62.0,71.0,1.0,pass,3,drawing,0,0,0,True,1,create,9,medium_block,NaN,None,NaN,None,False,False,7.74,52.84,3.0,sideway_left,22.99,3.0,hsr,-19.61,-19.39,22.17,25.71,3.54,32.89,33.11,0.22,NaN,None,False,13068.0,S. Lukić,12.0,LM,62.10,62.23,-179.76,-178.86,3.40,1.0,jogging,19.59,0.0,0.0,10.

In [9]:
on_ball_engagement["end_type"].value_counts()

end_type
indirect_regain        44470
indirect_disruption    10798
direct_regain           8447
direct_disruption       5424
foul_committed          3886
Name: count, dtype: int64

In [12]:
# OBE metrics aggregation
player_onball_metrics = (
    on_ball_engagement.groupby(["player_id", "player_name"])
    .agg(
        # Overall engagement
        obe_total_events=("event_id", "count"),
        obe_distance_total=("distance_covered", "sum"),
        obe_distance_avg=("distance_covered", "mean"),
        obe_speed_avg=("speed_avg", "mean"),
        
        # Line-breaking actions
        obe_linebreaks_affected=("affected_line_break", lambda x: x.notna().sum()),
        obe_linebreaks_attempted=("affected_line_breaking_passing_option_attempted", lambda x: x.eq(True).sum()),
        obe_linebreaks_xthreat=("affected_line_breaking_passing_option_xthreat", "sum"),
        obe_linebreaks_dangerous=("affected_line_breaking_passing_option_dangerous", lambda x: x.eq(True).sum()),

        # Pressing chain metrics
        obe_pressing_chains_total=("pressing_chain", lambda x: x.eq(True).sum()),
        obe_pressing_chain_length_avg=("pressing_chain_length", "mean"),
        obe_pressing_chain_direct_regain=("pressing_chain_end_type", lambda x: (x=="direct_regain").sum()),
        obe_pressing_chain_indirect_regain=("pressing_chain_end_type", lambda x: (x=="indirect_regain").sum()),
        obe_pressing_chain_direct_disruption=("pressing_chain_end_type", lambda x: (x=="direct_disruption").sum()),
        obe_pressing_chain_indirect_disruption=("pressing_chain_end_type", lambda x: (x=="indirect_disruption").sum()),
        obe_pressing_chain_foul=("pressing_chain_end_type", lambda x: (x=="foul_committed").sum()),

        # Defensive effect metrics
        obe_possession_losses_forced=("stop_possession_danger", "sum"),
        obe_beaten_by_possession=("beaten_by_possession", "sum"),
        obe_reduce_possession_danger=("reduce_possession_danger", "sum"),
        
        # Engagement type counts (optional, if you want to see which OBE type they do)
        obe_pressing_events=("event_subtype", lambda x: (x=="pressing").sum()),
        obe_pressure_events=("event_subtype", lambda x: (x=="pressure").sum()),
        obe_other_events=("event_subtype", lambda x: (x=="other").sum())
    )
    .reset_index()
)

player_onball_metrics.head()

,player_id,player_name,obe_total_events,obe_distance_total,obe_distance_avg,obe_speed_avg,obe_linebreaks_affected,obe_linebreaks_attempted,obe_linebreaks_xthreat,obe_linebreaks_dangerous,obe_pressing_chains_total,obe_pressing_chain_length_avg,obe_pressing_chain_direct_regain,obe_pressing_chain_indirect_regain,obe_pressing_chain_direct_disruption,obe_pressing_chain_indirect_disruption,obe_pressing_chain_foul,obe_possession_losses_forced,obe_beaten_by_possession,obe_reduce_possession_danger,obe_pressing_events,obe_pressure_events,obe_other_events
0,82,A. Cresswell,159,1165.16,7.328050,12.349796,3,2,0.0398,1,25,4.400000,0,0,0,0,0,6,1,9,21,39,62
1,124,A. Doucouré,2240,18677.69,8.338254,15.244330,264,105,1.8589,18,928,3.914871,0,0,0,0,0,26,25,21,622,599,327
2,168,A. Armstrong,703,6178.25,8.788407,15.101319,78,30,0.3188,1,315,3.990476,0,0,0,0,0,3,7,4,222,154,116
3,192,A. Lallana,244,1637.97,6.712992,13.840958,17,8,0.1409,1,61,3.901639,0,0,0,0,0,4,4,5,39,84,48
4,209,A. Smith,648,6534.92,10.084753,14.110640,21,15,0.2562,3,153,4.444444,0,0,0,0,0,18,24,14,99,187,190


# Shots

In [13]:
possession = events[events["event_type"] == "player_possession"].copy()

In [14]:
# Drop columns that are all NaN
possession = possession.dropna(axis=1, how='all')

pd.set_option('display.max_columns', None)  # ensure all columns are visible
possession.head()

,event_id,index,match_id,frame_start,frame_end,time_start,time_end,minute_start,second_start,duration,period,attacking_side_id,attacking_side,event_type_id,event_type,player_id,player_name,player_position_id,player_position,team_id,team_shortname,x_start,y_start,channel_id_start,channel_start,third_id_start,third_start,penalty_area_start,x_end,y_end,channel_id_end,channel_end,third_id_end,third_end,penalty_area_end,game_state_id,game_state,team_score,opponent_team_score,phase_index,player_possession_phase_index,first_player_possession_in_team_possession,last_player_possession_in_team_possession,lead_to_different_phase,issued_from_different_phase,n_player_possessions_in_phase,team_possession_loss_in_phase,team_in_possession_phase_type_id,team_in_possession_phase_type,team_out_of_possession_phase_type_id,team_out_of_possession_phase_type,current_team_in_possession_next_phase_type_id,current_team_in_possession_next_phase_type,current_team_out_of_possession_next_phase_type_id,current_team_out_of_possession_next_phase_type,current_team_in_possession_previous_phase_type_id,current_team_in_possession_previous_phase_type,current_team_out_of_possession_previous_phase_type_id,current_team_out_of_possession_previous_phase_type,game_interruption_before_id,game_interruption_before,game_interruption_after_id,game_interruption_after,lead_to_shot,lead_to_goal,distance_covered,trajectory_angle,trajectory_direction_id,trajectory_direction,in_to_out,out_to_in,speed_avg,speed_avg_band_id,speed_avg_band,separation_start,separation_end,separation_gain,last_defensive_line_x_start,last_defensive_line_x_end,delta_to_last_defensive_line_start,delta_to_last_defensive_line_end,delta_to_last_defensive_line_gain,last_defensive_line_height_start,last_defensive_line_height_end,last_defensive_line_height_gain,inside_defensive_shape_start,inside_defensive_shape_end,start_type_id,start_type,end_type_id,end_type,one_touch,quick_pass,carry,forward_momentum,is_header,hand_pass,initiate_give_and_go,pass_angle_received,pass_direction_received_id,pass_direction_received,pass_distance_received,pass_range_received_id,pass_range_received,pass_outcome_id,pass_outcome,targeted_passing_option_event_id,high_pass,player_targeted_id,player_targeted_name,player_targeted_position_id,player_targeted_position,player_targeted_x_pass,player_targeted_y_pass,player_targeted_channel_pass_id,player_targeted_channel_pass,player_targeted_third_pass_id,player_targeted_third_pass,player_targeted_penalty_area_pass,player_targeted_x_reception,player_targeted_y_reception,player_targeted_channel_reception_id,player_targeted_channel_reception,player_targeted_third_reception_id,player_targeted_third_reception,player_targeted_penalty_area_reception,player_targeted_xpass_completion,player_targeted_difficult_pass_target,player_targeted_xthreat,player_targeted_dangerous,n_passing_options,n_off_ball_runs,n_passing_options_line_break,n_passing_options_first_line_break,n_passing_options_second_last_line_break,n_passing_options_last_line_break,n_passing_options_ahead,n_passing_options_dangerous_difficult,n_passing_options_dangerous_not_difficult,n_passing_options_not_dangerous_not_difficult,n_passing_options_not_dangerous_difficult,n_passing_options_at_start,n_passing_options_at_end,n_passing_options_ahead_at_start,n_passing_options_ahead_at_end,n_teammates_ahead_end,n_teammates_ahead_start,organised_defense,defensive_structure,n_defensive_lines,first_line_break,second_last_line_break,last_line_break,furthest_line_break_id,furthest_line_break,furthest_line_break_type_id,furthest_line_break_type,interplayer_distance,interplayer_distance_range_id,interplayer_distance_range,interplayer_angle,interplayer_direction_id,interplayer_direction,pass_distance,pass_range_id,pass_range,pass_angle,pass_direction_id,pass_direction,pass_ahead,n_opponents_ahead_player_in_possession_pass_moment,n_opponents_ahead_pass_reception,n_opponents_bypassed,n_opponents_ahead_end,n_opponents_ahead_start,n_opponents_overtaken,is_pla

In [15]:
possession["end_type"].value_counts()

end_type
pass               329716
possession_loss     15666
shot                 8677
foul_suffered        4200
clearance            3733
unknown               861
Name: count, dtype: int64

In [16]:
shots = possession[possession["end_type"] == "shot"].copy()

In [19]:
import numpy as np

# =====================================================
# 1. PREP FEATURES (distance + angle)
# =====================================================

GOAL_X = 52.5
GOAL_Y = 0
GOAL_WIDTH = 7.32

# Shot distance
possession["shot_distance"] = np.sqrt(
    (GOAL_X - possession["x_end"])**2 +
    (GOAL_Y - possession["y_end"])**2
)

# Shot angle
possession["shot_angle"] = np.arctan2(
    GOAL_WIDTH * (GOAL_X - possession["x_end"]),
    (GOAL_X - possession["x_end"])**2 +
    (possession["y_end"])**2 -
    (GOAL_WIDTH / 2)**2
)

# =====================================================
# 2. PLAYER SHOT METRICS AGGREGATION
# =====================================================

player_shot_metrics = (
    possession.groupby(["player_id", "player_name"])
    .agg(
        # 🔢 Volume & output
        shots_total=("event_id", "count"),
        shots_goals=("lead_to_goal", "sum"),
        shots_conversion=("lead_to_goal", "mean"),

        # 📍 Location
        shots_avg_distance=("shot_distance", "mean"),
        shots_median_distance=("shot_distance", "median"),
        shots_avg_angle=("shot_angle", "mean"),

        shots_pct_inside_box=("penalty_area_end", "mean"),
        shots_pct_central=("channel_end", lambda x: (x == "center").mean()),
        shots_pct_half_space=("channel_end", lambda x: x.str.contains("half_space", na=False).mean()),
        shots_pct_wide=("channel_end", lambda x: x.str.contains("wide", na=False).mean()),

        # ⚡ Speed & transition
        shots_avg_speed=("speed_avg", "mean"),
        shots_pct_hsr=("speed_avg_band", lambda x: (x == "hsr").mean()),
        shots_pct_sprinting=("speed_avg_band", lambda x: (x == "sprinting").mean()),

        shots_pct_forward_momentum=("forward_momentum", "mean"),
        shots_pct_carry=("carry", "mean"),

        # 🧱 Pressure
        shots_avg_opponents_ahead=("n_opponents_ahead_end", "mean"),
        shots_avg_opponents_bypassed=("n_opponents_bypassed", "mean"),
        shots_avg_opponents_overtaken=("n_opponents_overtaken", "mean"),

        shots_pct_vs_low_block=("team_out_of_possession_phase_type", lambda x: (x == "low_block").mean()),
        shots_pct_vs_high_block=("team_out_of_possession_phase_type", lambda x: (x == "high_block").mean()),
        shots_pct_vs_transition=("team_out_of_possession_phase_type", lambda x: x.str.contains("transition", na=False).mean()),

        # 📐 Separation
        shots_avg_separation=("separation_end", "mean"),
        shots_avg_separation_gain=("separation_gain", "mean"),
        shots_pct_inside_def_shape=("inside_defensive_shape_end", "mean"),

        # 🧠 Finishing style
        shots_pct_one_touch=("one_touch", "mean"),
        shots_pct_headers=("is_header", "mean"),

        shots_pct_after_quick_pass=("quick_pass", "mean"),
        shots_pct_give_and_go=("initiate_give_and_go", "mean"),

        # 🔗 Build-up
        shots_pct_after_pass_reception=("start_type", lambda x: (x == "pass_reception").mean()),
        shots_pct_after_recovery=("start_type", lambda x: (x == "recovery").mean()),
        shots_pct_after_set_piece=("start_type", lambda x: x.str.contains("corner|free_kick", na=False).mean()),

        shots_avg_pass_length_before=("pass_distance_received", "mean"),
        shots_pct_long_pass_assist=("pass_range_received", lambda x: (x == "long").mean()),

        # 🧩 Phase
        shots_pct_finish_phase=("team_in_possession_phase_type", lambda x: (x == "finish").mean()),
        shots_pct_transition_phase=("team_in_possession_phase_type", lambda x: x.str.contains("transition", na=False).mean()),
        shots_pct_build_up_phase=("team_in_possession_phase_type", lambda x: (x == "build_up").mean()),

        shots_pct_first_in_phase=("first_player_possession_in_team_possession", "mean"),
        shots_pct_last_in_phase=("last_player_possession_in_team_possession", "mean"),

        # 🎯 Decision context
        shots_avg_passing_options=("n_passing_options", "mean"),
        shots_avg_passing_options_ahead=("n_passing_options_ahead", "mean"),

        # 🧬 Defensive line
        shots_avg_delta_def_line_end=("delta_to_last_defensive_line_end", "mean"),
        shots_avg_delta_def_line_gain=("delta_to_last_defensive_line_gain", "mean"),

        # 🧭 Direction
        shots_pct_forward_direction=("trajectory_direction", lambda x: (x == "forward").mean()),
        shots_pct_cut_inside=("in_to_out", "mean"),

        # 🧱 Defensive structure
        shots_pct_vs_organised_defense=("organised_defense", "mean"),
        shots_avg_defensive_lines=("n_defensive_lines", "mean"),
    )
    .reset_index()
)

# =====================================================
# 3. ADD METRICS REQUIRING MULTIPLE COLUMNS
# =====================================================

# Dangerous passing options (needs both columns)
player_shot_metrics["shots_avg_dangerous_options"] = (
    possession["n_passing_options_dangerous_difficult"].fillna(0)
    + possession["n_passing_options_dangerous_not_difficult"].fillna(0)
).groupby(possession["player_id"]).mean().values

# =====================================================
# 4. COMPOSITE METRICS
# =====================================================

# Pressure index
player_shot_metrics["shots_pressure_index"] = (
    player_shot_metrics["shots_avg_opponents_ahead"]
    - player_shot_metrics["shots_avg_separation"]
)

# Freedom index
player_shot_metrics["shots_freedom_index"] = (
    player_shot_metrics["shots_avg_separation"]
    / (player_shot_metrics["shots_avg_opponents_ahead"] + 1)
)

# Aggressiveness
player_shot_metrics["shots_aggressiveness"] = (
    player_shot_metrics["shots_total"]
    / (player_shot_metrics["shots_total"] + player_shot_metrics["shots_avg_passing_options_ahead"])
)

# Transition tendency
player_shot_metrics["shots_transition_pct"] = (
    player_shot_metrics["shots_pct_forward_momentum"]
)

# Self-created shots
player_shot_metrics["shots_self_created_pct"] = (
    player_shot_metrics["shots_pct_carry"]
    + player_shot_metrics["shots_pct_forward_momentum"]
)

# Shot difficulty index (improved)
player_shot_metrics["shots_difficulty_index"] = (
    player_shot_metrics["shots_avg_distance"]
    + player_shot_metrics["shots_avg_opponents_ahead"]
    - player_shot_metrics["shots_avg_separation"]
)

# =====================================================
# DONE
# =====================================================

player_shot_metrics.head()

,player_id,player_name,shots_total,shots_goals,shots_conversion,shots_avg_distance,shots_median_distance,shots_avg_angle,shots_pct_inside_box,shots_pct_central,shots_pct_half_space,shots_pct_wide,shots_avg_speed,shots_pct_hsr,shots_pct_sprinting,shots_pct_forward_momentum,shots_pct_carry,shots_avg_opponents_ahead,shots_avg_opponents_bypassed,shots_avg_opponents_overtaken,shots_pct_vs_low_block,shots_pct_vs_high_block,shots_pct_vs_transition,shots_avg_separation,shots_avg_separation_gain,shots_pct_inside_def_shape,shots_pct_one_touch,shots_pct_headers,shots_pct_after_quick_pass,shots_pct_give_and_go,shots_pct_after_pass_reception,shots_pct_after_recovery,shots_pct_after_set_piece,shots_avg_pass_length_before,shots_pct_long_pass_assist,shots_pct_finish_phase,shots_pct_transition_phase,shots_pct_build_up_phase,shots_pct_first_in_phase,shots_pct_last_in_phase,shots_avg_passing_options,shots_avg_passing_options_ahead,shots_avg_delta_def_line_end,shots_avg_delta_def_line_gain,shots_pct_forward_direction,shots_pct_cut_inside,shots_pct_vs_organised_defense,shots_avg_defensive_lines,shots_avg_dangerous_options,shots_pressure_index,shots_freedom_index,shots_aggressiveness,shots_transition_pct,shots_self_created_pct,shots_difficulty_index
0,82,A. Cresswell,531,2,0.003766,66.909366,68.420264,0.106530,0.001883,0.105461,0.227872,0.666667,7.776844,0.024482,0.003766,0.0,0.410546,9.050847,2.034908,-0.007533,0.193974,0.212806,0.007533,5.452994,-2.570716,0.071563,0.220339,0.043315,0.303202,0.022599,0.794727,0.045198,0.009416,14.688910,0.037665,0.193974,0.007533,0.212806,0.152542,0.173258,2.489642,1.638418,26.014953,-0.632392,0.274953,0.114679,0.706366,2.726744,0.145009,3.597853,0.542541,0.996924,0.0,0.410546,70.507220
1,124,A. Doucouré,972,7,0.007202,50.539543,48.492562,0.150523,0.059671,0.234568,0.343621,0.421811,10.570521,0.075103,0.014403,0.010288,0.453704,6.258230,0.041766,-0.288066,0.294239,0.055556,0.063786,3.177263,-1.014805,0.605967,0.338477,0.054527,0.182099,0.103909,0.683128,0.100823,0.006173,11.597229,0.019547,0.294239,0.063786,0.055556,0.221193,0.235597,2.435185,0.916667,14.895741,-0.190700,0.307613,0.117914,0.385442,2.721362,0.402263,3.080967,0.437746,0.999058,0.010288,0.463992,53.620510
2,168,A. Armstrong,337,7,0.020772,44.681064,42.070286,0.165883,0.127596,0.207715,0.284866,0.507418,11.345174,0.083086,0.023739,0.032641,0.540059,4.940653,-1.148855,-0.626113,0.409496,0.035608,0.071217,2.895163,-1.272493,0.409496,0.278932,0.047478,0.154303,0.127596,0.762611,0.047478,0.014837,15.661012,0.077151,0.409496,0.071217,0.035608,0.139466,0.261128,2.359050,0.664688,10.146766,1.039941,0.350148,0.148352,0.412214,2.740741,0.578635,2.045490,0.487348,0.998032,0.032641,0.5727,46.726554
3,192,A. Lallana,274,2,0.007299,55.990844,53.186209,0.137752,0.018248,0.295620,0.463504,0.240876,10.803258,0.021898,0.003650,0.040146,0.49635,7.481752,0.721311,-0.105839,0.240876,0.171533,0.018248,2.839562,-1.335401,0.613139,0.321168,0.036496,0.20073,0.047445,0.799270,0.047445,0.025547,11.500685,0.021898,0.240876,0.018248,0.171533,0.135036,0.171533,2.868613,1.394161,18.451241,-0.668394,0.229927,0.183824,0.528689,2.914729,0.401460,4.642190,0.334785,0.994938,0.040146,0.536496,60.633033
4,209,A. Smith,567,3,0.005291,58.995960,58.096245,0.111796,0.008818,0.091711,0.153439,0.754850,9.236272,0.051146,0.010582,0.0,0.352734,8.195767,1.244681,-0.003527,0.322751,0.068783,0.021164,4.611711,-1.956067,0.104056,0.33157,0.091711,0.253968,0.061728,0.671958,0.084656,0.019400,15.227638,0.044092,0.322751,0.021164,0.068783,0.275132,0.262787,2.305115,1.347443,21.682328,-0.323175,0.354497,0.15,0.57234,2.620818,0.232804,3.584056,0.501504,0.997629,0.0,0.352734,62.580017


## Other possessions

In [20]:
possession = events[events["event_type"] == "player_possession"].copy()
possession.head()

,event_id,index,match_id,frame_start,frame_end,frame_physical_start,time_start,time_end,minute_start,second_start,duration,period,attacking_side_id,attacking_side,event_type_id,event_type,event_subtype_id,event_subtype,player_id,player_name,player_position_id,player_position,player_in_possession_id,player_in_possession_name,player_in_possession_position_id,player_in_possession_position,team_id,team_shortname,x_start,y_start,channel_id_start,channel_start,third_id_start,third_start,penalty_area_start,x_end,y_end,channel_id_end,channel_end,third_id_end,third_end,penalty_area_end,associated_player_possession_event_id,associated_player_possession_frame_start,associated_player_possession_frame_end,associated_player_possession_end_type_id,associated_player_possession_end_type,associated_off_ball_run_event_id,associated_off_ball_run_subtype_id,associated_off_ball_run_subtype,game_state_id,game_state,team_score,opponent_team_score,phase_index,player_possession_phase_index,first_player_possession_in_team_possession,last_player_possession_in_team_possession,lead_to_different_phase,issued_from_different_phase,n_player_possessions_in_phase,team_possession_loss_in_phase,team_in_possession_phase_type_id,team_in_possession_phase_type,team_out_of_possession_phase_type_id,team_out_of_possession_phase_type,current_team_in_possession_next_phase_type_id,current_team_in_possession_next_phase_type,current_team_out_of_possession_next_phase_type_id,current_team_out_of_possession_next_phase_type,current_team_in_possession_previous_phase_type_id,current_team_in_possession_previous_phase_type,current_team_out_of_possession_previous_phase_type_id,current_team_out_of_possession_previous_phase_type,game_interruption_before_id,game_interruption_before,game_interruption_after_id,game_interruption_after,lead_to_shot,lead_to_goal,distance_covered,trajectory_angle,trajectory_direction_id,trajectory_direction,in_to_out,out_to_in,speed_avg,speed_avg_band_id,speed_avg_band,separation_start,separation_end,separation_gain,last_defensive_line_x_start,last_defensive_line_x_end,delta_to_last_defensive_line_start,delta_to_last_defensive_line_end,delta_to_last_defensive_line_gain,last_defensive_line_height_start,last_defensive_line_height_end,last_defensive_line_height_gain,inside_defensive_shape_start,inside_defensive_shape_end,start_type_id,start_type,end_type_id,end_type,consecutive_on_ball_engagements,one_touch,quick_pass,carry,forward_momentum,is_header,hand_pass,initiate_give_and_go,pass_angle_received,pass_direction_received_id,pass_direction_received,pass_distance_received,pass_range_received_id,pass_range_received,pass_outcome_id,pass_outcome,targeted_passing_option_event_id,high_pass,player_targeted_id,player_targeted_name,player_targeted_position_id,player_targeted_position,player_targeted_x_pass,player_targeted_y_pass,player_targeted_channel_pass_id,player_targeted_channel_pass,player_targeted_third_pass_id,player_targeted_third_pass,player_targeted_penalty_area_pass,player_targeted_x_reception,player_targeted_y_reception,player_targeted_channel_reception_id,player_targeted_channel_reception,player_targeted_third_reception_id,player_targeted_third_reception,player_targeted_penalty_area_reception,player_targeted_distance_to_goal_start,player_targeted_distance_to_goal_end,player_targeted_angle_to_goal_start,player_targeted_angle_to_goal_end,player_targeted_average_speed,player_targeted_speed_avg_band_id,player_targeted_speed_avg_band,speed_difference,player_targeted_xpass_completion,player_targeted_difficult_pass_target,player_targeted_xthreat,player_targeted_dangerous,n_passing_options,n_off_ball_runs,n_passing_options_line_break,n_passing_options_first_line_break,n_passing_options_second_last_line_break,n_passing_options_last_line_break,n_passing_options_ahead,n_passing_options_dangerous_difficult,n_passing_options_dangerous_not_difficult,n_passing_options_not_dangerous_not_difficult,n_passing_options_not_dangerous_difficult,n_passing_options_at_start,n_passi

In [22]:
player_possession_metrics = (
    possession.groupby(["player_id", "player_name"])
    .agg(
        # -------------------------
        # VOLUME
        # -------------------------
        possessions_total=("event_id", "count"),

        # -------------------------
        # END TYPE DISTRIBUTION
        # -------------------------
        possessions_ending_shot=("end_type", lambda x: (x == "shot").sum()),
        possessions_ending_pass=("end_type", lambda x: (x == "pass").sum()),
        possessions_ending_loss=("end_type", lambda x: (x == "possession_loss").sum()),
        possessions_ending_foul_suffered=("end_type", lambda x: (x == "foul_suffered").sum()),
        possessions_ending_clearance=("end_type", lambda x: (x == "clearance").sum()),

        # -------------------------
        # EFFICIENCY
        # -------------------------
        possession_shot_rate=("end_type", lambda x: (x == "shot").mean()),
        possession_loss_rate=("end_type", lambda x: (x == "possession_loss").mean()),

        # -------------------------
        # DURATION & DISTANCE
        # -------------------------
        possession_duration_avg=("duration", "mean"),
        possession_duration_total=("duration", "sum"),

        possession_distance_avg=("distance_covered", "mean"),
        possession_distance_total=("distance_covered", "sum"),

        # -------------------------
        # SPEED & INTENSITY
        # -------------------------
        possession_speed_avg=("speed_avg", "mean"),
        possession_high_speed_actions=("speed_avg_band", lambda x: x.isin(["hsr", "sprinting"]).sum()),

        # -------------------------
        # PROGRESSION
        # -------------------------
        possession_forward_actions=("forward_momentum", lambda x: x.eq(True).sum()),
        possession_forward_rate=("forward_momentum", lambda x: x.eq(True).mean()),

        possession_progression_avg=("delta_to_last_defensive_line_end", "mean"),
        possession_progression_total=("delta_to_last_defensive_line_end", "sum"),

        # -------------------------
        # DEFENSIVE LINE BREAKING
        # -------------------------
        possession_line_breaks=("n_opponents_bypassed", "sum"),
        possession_line_breaks_avg=("n_opponents_bypassed", "mean"),

        # -------------------------
        # PRESSURE / SPACE
        # -------------------------
        possession_separation_start_avg=("separation_start", "mean"),
        possession_separation_end_avg=("separation_end", "mean"),
        possession_separation_gain_avg=("separation_gain", "mean"),

        # -------------------------
        # POSITIONAL CONTEXT
        # -------------------------
        possession_attacking_third_rate=("third_start", lambda x: (x == "attacking_third").mean()),
        possession_penalty_area_rate=("penalty_area_end", lambda x: x.eq(True).mean()),

        # -------------------------
        # ACTION TYPES
        # -------------------------
        possession_carries=("carry", lambda x: x.eq(True).sum()),
        possession_carry_rate=("carry", lambda x: x.eq(True).mean()),

        possession_one_touch=("one_touch", lambda x: x.eq(True).sum()),
        possession_one_touch_rate=("one_touch", lambda x: x.eq(True).mean()),

        possession_headers=("is_header", lambda x: x.eq(True).sum()),

        # -------------------------
        # PASS CONTEXT
        # -------------------------
        possession_received_pass_avg_distance=("pass_distance_received", "mean"),
        possession_received_pass_forward_rate=("pass_direction_received", lambda x: (x == "forward").mean()),

        # -------------------------
        # TEAM CONTEXT
        # -------------------------
        possession_teammates_ahead_avg=("n_teammates_ahead_end", "mean"),
        possession_opponents_ahead_avg=("n_opponents_ahead_end", "mean"),

        # -------------------------
        # GAME STATE
        # -------------------------
        possession_winning_rate=("game_state", lambda x: (x == "winning").mean()),
        possession_drawing_rate=("game_state", lambda x: (x == "drawing").mean()),
        possession_losing_rate=("game_state", lambda x: (x == "losing").mean()),

        # -------------------------
        # PHASE CONTEXT
        # -------------------------
        possession_finish_phase=("team_in_possession_phase_type", lambda x: (x == "finish").sum()),
        possession_build_up_phase=("team_in_possession_phase_type", lambda x: (x == "build_up").sum()),
        possession_create_phase=("team_in_possession_phase_type", lambda x: (x == "create").sum()),

        # -------------------------
        # DIRECTNESS / STYLE
        # -------------------------
        possession_verticality=("trajectory_angle", "mean"),
    )
    .reset_index()
)

player_possession_metrics.head()

,player_id,player_name,possessions_total,possessions_ending_shot,possessions_ending_pass,possessions_ending_loss,possessions_ending_foul_suffered,possessions_ending_clearance,possession_shot_rate,possession_loss_rate,possession_duration_avg,possession_duration_total,possession_distance_avg,possession_distance_total,possession_speed_avg,possession_high_speed_actions,possession_forward_actions,possession_forward_rate,possession_progression_avg,possession_progression_total,possession_line_breaks,possession_line_breaks_avg,possession_separation_start_avg,possession_separation_end_avg,possession_separation_gain_avg,possession_attacking_third_rate,possession_penalty_area_rate,possession_carries,possession_carry_rate,possession_one_touch,possession_one_touch_rate,possession_headers,possession_received_pass_avg_distance,possession_received_pass_forward_rate,possession_teammates_ahead_avg,possession_opponents_ahead_avg,possession_winning_rate,possession_drawing_rate,possession_losing_rate,possession_finish_phase,possession_build_up_phase,possession_create_phase,possession_verticality
0,82,A. Cresswell,531,2,517,6,0,5,0.003766,0.011299,1.288701,684.3,3.016855,1601.95,7.776844,15,0,0.000000,26.014953,13813.94,991.0,2.034908,8.023879,5.452994,-2.570716,0.120527,0.001883,218,0.410546,117,0.220339,23,14.688910,0.048964,6.789077,9.050847,0.290019,0.461394,0.248588,103,113,227,2.354754
1,124,A. Doucouré,972,29,879,54,0,7,0.029835,0.055556,1.228189,1193.8,4.108354,3993.32,10.570521,87,10,0.010288,14.895741,14478.66,35.0,0.041766,4.192335,3.177263,-1.014805,0.341564,0.059671,441,0.453704,329,0.338477,53,11.597229,0.237654,3.414609,6.258230,0.283951,0.525720,0.190329,286,54,360,0.360731
2,168,A. Armstrong,337,26,278,24,2,4,0.077151,0.071217,1.492285,502.9,5.023976,1693.08,11.345174,36,11,0.032641,10.146766,3419.46,-301.0,-1.148855,4.167745,2.895163,-1.272493,0.483680,0.127596,182,0.540059,94,0.278932,16,15.661012,0.362018,2.448071,4.940653,0.053412,0.483680,0.462908,138,12,102,-1.584792
3,192,A. Lallana,274,1,253,15,5,0,0.003650,0.054745,1.180657,323.5,3.803869,1042.26,10.803258,7,11,0.040146,18.451241,5055.64,176.0,0.721311,4.175182,2.839562,-1.335401,0.237226,0.018248,136,0.496350,88,0.321168,10,11.500685,0.313869,4.919708,7.481752,0.142336,0.233577,0.624088,66,47,107,-3.376606
4,209,A. Smith,567,5,531,16,2,13,0.008818,0.028219,1.073545,608.7,3.025009,1715.18,9.236272,35,0,0.000000,21.682328,12293.88,585.0,1.244681,6.567390,4.611711,-1.956067,0.236332,0.008818,200,0.352734,188,0.331570,52,15.227638,0.070547,5.791887,8.195767,0.308642,0.483245,0.208113,183,39,188,-1.285204
